# Part 2a – execution times overview

Durations are read from the **`real`** line of the `time` command output in each log file (wall-clock time).

1. **Table 1:** Raw **wall-clock** runtime for each workload × interference configuration (shown as **whole seconds**; no fractional part after `m` / before `s`).
2. **Table 2:** **Normalized runtime** per workload: \(T_{\text{config}} / T_{\text{none}}\). The **none** column is the baseline (**1.00**). Color coding: **green** (#009B47) if ≤ 1.3, **orange** (#F7981A) if &gt; 1.3 and ≤ 2, **red** (#E91D21) if &gt; 2 (white text on colored cells).
3. **Run list:** One line per experiment, e.g. `barnes-none: 2m3.235s` (verbatim from `real`; run the last code cell after the tables).


In [14]:
from pathlib import Path

import numpy as np
import pandas as pd
import re
from IPython.display import display, HTML


WORKLOADS = [
    "barnes",
    "blackscholes",
    "canneal",
    "freqmine",
    "radix",
    "streamcluster",
    "vips",
]
CONFIG_ORDER = ["cpu", "l1d", "l1i", "l2", "llc", "membw"]


def resolve_result_dir() -> Path:
    root = Path.cwd()
    if (root / "part2a_results" / "barnes_none.txt").exists():
        return root / "part2a_results"
    if (root / "barnes_none.txt").exists():
        return root
    raise FileNotFoundError(
        "Could not find barnes_none.txt. Run from the repo root or set cwd to part2a_results."
    )


RESULT_DIR = resolve_result_dir()

COL_REAL = ["none"] + CONFIG_ORDER
COL_DISP = {
    "none": "none",
    "cpu": "cpu",
    "l1d": "l1d",
    "l1i": "l1i",
    "l2": "l2",
    "llc": "llc",
    "membw": "memBW",
}


def parse_real_seconds(text: str) -> float:
    for line in text.splitlines():
        if line.startswith("real"):
            m = re.search(r"(\d+)m([\d.]+)s", line)
            if m:
                return int(m.group(1)) * 60.0 + float(m.group(2))

    raise ValueError("No matching 'real … XmY.YYs' line found")


def build_seconds_frame() -> pd.DataFrame:
    rows = []
    for wl in WORKLOADS:
        row = {"Workload": wl}
        for key in COL_REAL:
            path = RESULT_DIR / f"{wl}_{key}.txt"
            if not path.exists():
                row[COL_DISP[key]] = np.nan
                continue
            row[COL_DISP[key]] = parse_real_seconds(path.read_text())
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_real(sec: float) -> str:
    if sec != sec:
        return "—"
    sec = float(sec)
    total = int(round(sec))
    if total == 0 and sec > 0:
        total = 1  
    m, s = divmod(total, 60)
    return f"{int(m)}m{s}s"


df_sec = build_seconds_frame()
raw_display_cols = ["Workload"] + [COL_DISP[k] for k in COL_REAL]
df_raw_str = df_sec.copy()
for c in raw_display_cols[1:]:
    df_raw_str[c] = df_sec[c].map(fmt_real)

display(HTML("<h3>1) Wall-clock (<code>real</code>) times</h3>"))
t1 = df_raw_str[raw_display_cols].style.set_table_styles(
    [
        {
            "selector": "th",
            "props": [
                ("font-weight", "bold"),
                ("color", "#111111"),
                ("background-color", "#ffffff"),
                ("border", "1px solid #444"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("border", "1px solid #444"),
                ("color", "#111111"),
                ("background-color", "#ffffff"),
            ],
        },
    ]
)
display(t1)


base = df_sec.set_index("Workload")[COL_DISP["none"]]
norm = pd.DataFrame({"Workload": df_sec["Workload"]})
for key in COL_REAL:
    c = COL_DISP[key]
    if key == "none":
        norm[c] = 1.0
    else:
        norm[c] = df_sec[c].to_numpy(dtype=float) / base.to_numpy(dtype=float)


def row_highlight(row: pd.Series):
    neutral = "background-color: #ffffff; color: #111111"
    style_workload = "background-color: #fafafa; color: #111111"
    dark_green = "background-color: #009B47; color: #ffffff"
    dark_orange = "background-color: #F7981A; color: #ffffff"
    dark_red = "background-color: #E91D21; color: #ffffff"
    na_style = "background-color: #f3f4f6; color: #6b7280"
    styles = []
    for col in norm.columns:
        if col == "Workload":
            styles.append(style_workload)
            continue
        if col == "none":
            styles.append(neutral)
            continue
        v = row[col]
        if pd.isna(v):
            styles.append(na_style)
        elif v <= 1.3:
            styles.append(dark_green)
        elif v <= 2.0:
            styles.append(dark_orange)
        else:
            styles.append(dark_red)
    return styles


fmt_cols = [c for c in norm.columns if c != "Workload"]
styled_norm = (
    norm.style.apply(row_highlight, axis=1)
    .format({c: "{:.2f}" for c in fmt_cols}, na_rep="—")
    .set_table_styles(
        [
            {
                "selector": "th",
                "props": [
                    ("font-weight", "bold"),
                    ("color", "#111111"),
                    ("background-color", "#ffffff"),
                    ("border", "1px solid #444"),
                ],
            },
            {
                "selector": "td",
                "props": [("border", "1px solid #444")],
            },
        ]
    )
)

display(HTML("<h3>2) Normalized time (config / none)</h3>"))
display(styled_norm)



,Workload,none,cpu,l1d,l1i,l2,llc,memBW
0,barnes,2m3s,2m58s,2m57s,3m0s,3m6s,3m21s,3m15s
1,blackscholes,0m1s,0m1s,0m1s,0m1s,0m1s,0m1s,0m1s
2,canneal,0m13s,0m14s,0m15s,0m21s,0m18s,0m19s,0m18s
3,freqmine,0m5s,0m10s,0m7s,0m10s,0m7s,0m9s,0m9s
4,radix,0m49s,0m39s,0m41s,0m42s,0m44s,0m44s,0m53s
5,streamcluster,0m7s,0m8s,0m9s,0m10s,0m8s,0m14s,0m14s
6,vips,1m19s,1m48s,1m40s,1m45s,2m6s,1m56s,1m53s


,Workload,none,cpu,l1d,l1i,l2,llc,memBW
0,barnes,1.00,1.44,1.44,1.46,1.51,1.63,1.58
1,blackscholes,1.00,1.42,1.42,1.49,1.48,1.60,1.72
2,canneal,1.00,1.07,1.16,1.68,1.45,1.51,1.44
3,freqmine,1.00,2.08,1.34,2.11,1.39,1.76,1.77
4,radix,1.00,0.80,0.85,0.85,0.91,0.90,1.08
5,streamcluster,1.00,1.12,1.34,1.49,1.20,1.97,1.96
6,vips,1.00,1.37,1.27,1.33,1.60,1.48,1.43


### 3) All experiment runs (wall-clock)

Ordered list: each **log file** (`workload_config.txt`) with **`real`** time from `time` output. Rounded display matches Table 1; the value in parentheses is the exact parsed duration in seconds.

**Requires:** run the first code cell first so `RESULT_DIR`, `WORKLOADS`, `COL_REAL`, `COL_DISP`, `parse_real_seconds`, and `fmt_real` are defined.


In [15]:
from IPython.display import display, HTML

order_pairs = [(wl, k) for wl in WORKLOADS for k in COL_REAL]
list_items = []
for wl, k in order_pairs:
    path = RESULT_DIR / f"{wl}_{k}.txt"
    if not path.exists():
        continue
    sec = parse_real_seconds(path.read_text())
    cfg_label = COL_DISP[k]
    human = fmt_real(sec)
    list_items.append(
        f'<li><code>{path.name}</code> — workload <strong>{wl}</strong>, interference '
        f'<strong>{cfg_label}</strong> — real <strong>{human}</strong> '
        f'<span style="color:#666">({sec:.3f} s)</span></li>'
    )

html = "<ul>" + "".join(list_items) + "</ul>"
display(HTML(html))
print(f"Listed {len(list_items)} runs from {RESULT_DIR.resolve()}")


Listed 49 runs from /Users/banu/Desktop/cloud-comp-arch-project/part2a_results
